# 클러스터링·증강: 상세 원리, 시각화, 검증

이 노트북은 실행 함수를 정의합니다. **00 통합 실행** 또는 **통합본_전체코드_시각화_결과**에서 실제로 호출합니다.

## 클러스터링을 왜 하는가?

정상 거래가 매우 많을 때 비슷한 거래를 묶은 뒤 각 묶음에서 표본을 뽑아 학습량을 줄입니다. 이상 거래는 전부 유지합니다. `MiniBatchKMeans`는 데이터를 작은 묶음으로 처리하며 중심점을 갱신합니다. K는 실제 데이터 기본 32개이며, 정답을 입력받아 군집을 만드는 지도학습이 아닙니다. 군집 적합 대상으로 정상 학습 거래를 고르는 데만 레이블을 사용합니다.

1. **적합 데이터:** 학습 날짜별 정상 거래에서 최대 4,096건을 고정 시드로 추출합니다. 검증·테스트는 제외합니다.
2. **입력:** 이번 독립 버전에서는 거래 특징 17개입니다. `StandardScaler`를 학습 참조 표본에만 적합한 뒤 평균 0·표준편차 1 기준으로 변환합니다.
3. **군집 적합:** `MiniBatchKMeans(n_clusters=K, batch_size=8192, n_init=3, random_state=seed)`를 사용합니다.
4. **일별 예산:** 정상 건수가 N, 비어 있지 않은 군집 수가 C, 목표 비율이 r이면 `B = min(N, max(C, ceil(N*r)))`입니다. 각 군집에 최소 1건을 배정하고 남은 예산을 **크기 비례 75% + 균등 25%**의 가중치로 배분합니다. 군집 크기보다 많이 뽑지 않도록 상한과 정수 반올림을 처리합니다. 따라서 최종 배분이 정확한 75:25 건수 분할이라는 뜻은 아닙니다.
5. **보정:** 군집 원본이 N_k건이고 n_k건을 뽑았다면 포함확률은 n_k/N_k, IPW는 N_k/n_k입니다. 이상 거래의 IPW는 1입니다. 손실에 이 값을 적용하고 표본수/원본수로 정규화합니다.
6. **문맥 유지:** 표본으로 고르는 것은 학습할 대상 거래입니다. 주변 계좌와 연결을 담은 일별 그래프는 보존하고, 배치에서 필요한 2홉 이웃을 사용합니다.

작은 예제에서는 군집별 최소 1건 때문에 목표 1%보다 훨씬 많이 뽑을 수 있습니다. 군집 수가 정상 거래 수보다 많으면 줄여야 합니다. 정상 패턴의 다양성을 보존하려는 목적이며, 군집화가 성능 향상을 보장하지는 않습니다.

### 시각화와 감사 지표 읽기

- PCA 그림: 17차원을 2차원으로 줄인 **일부 정상 거래**와 추출 거래입니다. PCA는 학습 참조 데이터에만 적합합니다. 그림에서 겹친다고 원래 17차원에서 같다는 뜻은 아닙니다.
- 막대 그림: 군집별 원본 건수와 표본 건수를 로그 축으로 비교합니다.
- KL: 학습 참조값의 분위수 구간을 사용해 `원본 P → 표본 Q`의 차이를 피처별로 계산합니다. 빈 구간에는 1e-8을 더합니다. raw, IPW 보정, 같은 정상 표본 수의 랜덤 추출을 비교합니다. **단변량 KL이 작다고 전체 관계가 보존되거나 탐지 성능이 좋다는 뜻은 아닙니다.**
- 거래 연결 그림: 첫 학습 날짜에서 뽑힌 정상 거래 최대 30건의 계좌·방향 연결입니다. 원형 배치는 보기 위한 배치이고, 실제 계좌 위치나 군집 간 거리를 의미하지 않습니다. 자기 송금은 이 배치에서 잘 보이지 않을 수 있습니다.

## 데이터 증강은 무엇을 만드는가?

**원본 거래 CSV나 계좌 연결을 합성하지 않습니다.** 학습된 GNN 두 층과 기존 분류기의 Linear/ReLU 은닉 투영을 고정한 뒤 거래 표현을 추출합니다. 원래 실험의 표현은 128차원이며 이 노트북의 빠른 데모는 hidden=16이므로 16차원입니다. 각 모델은 자기 모델이 만든 표현을 사용합니다.

| 조건 | 만드는 데이터 | 비교 목적 |
|---|---|---|
| none | 새 데이터 없음 | 같은 고정 인코더·새 분류기 절차의 기준 |
| repeat | 기존 이상 거래 표현 반복 추출 | 단순 반복 효과 |
| smote | 가까운 이상 표현 두 개 사이를 보간 | 보간 생성 효과 |
| wgan_gp | 잡음에서 학습한 이상 표현 생성 | 신경망 생성 효과 |

### SMOTE

학습 양성 표현 x_i의 가까운 이웃 x_j를 찾고 `x_new = x_i + λ(x_j-x_i)`로 만듭니다. λ는 0~1 난수입니다. 자기 자신을 이웃에서 제외하고 기본 k=5, 표본이 작으면 가능한 이웃 수까지 줄입니다. 생성 결과가 실제로 해당 선분 위에 있는지 검사합니다. 범주형 계좌 정보나 거래 원본 열을 직접 보간하는 것이 아닙니다.

### WGAN-GP

생성기: `32차원 잡음 → 128 → 128 → 표현 차원`.
Critic: `표현 차원 → 128 → 128 → 1`.
Critic은 진짜/가짜의 점수 차이를 학습하고, 입력 보간점에서 기울기 크기가 1에 가까워지도록 `10 × (기울기 크기−1)²`를 더합니다. Critic 3번 갱신 후 생성기를 1번 갱신합니다. 학습 이상 표현으로만 표준화·적합합니다.

**손실 곡선만으로 생성 품질을 판단하지 않습니다.** 분산 비율, 가장 가까운 학습 표현까지의 거리, 거의 같은 표본의 비율도 기록합니다. `possible_collapse`는 낮은 분산/과도한 복제를 찾는 휴리스틱이며 품질 인증이 아닙니다. repeat는 원래 복제 방식이므로 해당 경고가 나오는 것이 자연스럽습니다. 데모의 짧은 GAN 학습은 동작 확인용입니다.

### 공정한 비교와 자원 한도

모든 조건은 같은 인코더·훈련/검증 표현·새 분류기 초기값·optimizer·학습 스텝을 사용합니다. repeat/SMOTE/GAN의 추가 표본 수도 같습니다. 양성 데이터가 늘었다는 이유로 손실의 양성 비중까지 늘지 않도록 `(정상 평균 손실 + α×이상 평균 손실)/(1+α)`를 사용합니다. α는 원래 보정된 양성/정상 질량과 pos_weight로 계산합니다.

분류기는 `표현 차원 → 128 → 1`입니다. **다른 컴퓨터에서도 실행 가능하도록 날짜·클래스별 표현 추출에 기본 2,048건 상한**을 둡니다. 학습 표본의 기존 IPW와 추가 추출 확률을 결합하고, 검증도 추출 확률의 역수로 보정합니다. 전체 거래 평가가 아니라 이 표본 기반 검증 추정값이며, 동일한 검증 표본을 모든 증강 조건이 공유합니다. 데모는 작아서 상한에 걸리지 않습니다. 전체 검증이 필요하면 상한을 늘리고 메모리 사용량을 고려하세요.

**기존 연구와의 차이:** 기존 군집 적합은 18개 거래 특징, 기존 잠재공간 증강은 고정 클러스터 표본과 해당 검증 표본을 사용했습니다. 이 독립 버전은 17개 거래 특징과 새로운 표본·검증 절차를 사용합니다. 아래에 함께 제공하는 기존 실험 결과와 새 실행 결과를 섞어 평균 내거나 동일 실험의 재현값으로 취급하지 마세요.


In [ ]:
from pathlib import Path
import sys, json
# 통합 노트북은 이 폴더를 작업 폴더로 지정합니다. 개별 실행도 같은 폴더에서 하세요.
WORKSPACE = Path.cwd().resolve()
RUNTIME = WORKSPACE / '_runtime'
(RUNTIME / 'gnn_pipeline').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'gnn_pipeline' / '__init__.py').touch()
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))


## 코드 셀을 내부 모듈에 연결하는 함수

In [ ]:
def export_cells(notebook_name, module_name):
    """이 노트북의 실제 코드 셀을 모아 내부 import용 모듈을 생성합니다."""
    notebook = json.loads((WORKSPACE / notebook_name).read_text(encoding='utf-8'))
    blocks = [cell['source'] for cell in notebook['cells']
              if cell['cell_type'] == 'code' and cell.get('metadata', {}).get('module') == module_name]
    source = '\n\n'.join(''.join(block) if isinstance(block, list) else block for block in blocks) + '\n'
    path = RUNTIME / 'gnn_pipeline' / (module_name + '.py')
    compile(source, str(path), 'exec')
    path.write_text(source, encoding='utf-8')
    print('내부 모듈 준비:', path.name)
    return path


## 라이브러리 준비

In [ ]:
"""Cluster audit, latent synthesis, matched-head comparisons, and visual checks."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_score, recall_score, f1_score
from gnn_pipeline import data
from gnn_pipeline.models import TransactionGNN
from gnn_pipeline.pipeline import read_graph, graph_dates
from gnn_pipeline.allocation import allocate



## `cluster_audit`

학습 데이터로만 군집과 PCA를 적합하고 첫 학습일의 표본을 실제로 추출합니다. 군집 건수·PCA·분포 KL·실제 거래 연결을 그리고 검사 결과를 반환합니다.

**입력:** `settings`, `cfg`

**반환:** `(table, audit, checks)`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def cluster_audit(settings, cfg):
    """Reproduce training-only clustering and audit one day's sampled population."""
    root = Path(cfg.output).parent / 'analysis'
    root.mkdir(parents=True, exist_ok=True)
    dates = graph_dates(cfg, 'train')
    reference = []
    for index, date in enumerate(dates):
        day = read_graph(cfg, date)
        negative = np.flatnonzero(day.labels == 0)
        rng = np.random.default_rng(cfg.seed + index)
        reference.append(day.target_x[rng.choice(negative, min(4096, len(negative)), replace=False)])
    reference = np.concatenate(reference)
    k = settings['clusters']
    if len(reference) < k:
        raise ValueError('Training negatives fewer than clusters; decrease clusters')
    scaler = StandardScaler().fit(reference)
    km = MiniBatchKMeans(n_clusters=k, random_state=cfg.seed, batch_size=8192, n_init=3).fit(scaler.transform(reference))
    day = read_graph(cfg, dates[0])
    negative = np.flatnonzero(day.labels == 0)
    labels = km.predict(scaler.transform(day.target_x[negative]))
    counts = np.bincount(labels, minlength=k)
    budgets = allocate(counts, settings['negative_rate'])
    rng = np.random.default_rng(cfg.seed)
    # Match prepare_samples: positives first, then independently draw each cluster.
    selected = np.concatenate([rng.choice(np.flatnonzero(labels == group), int(size), replace=False)
                               for group, size in enumerate(budgets) if size])
    ipw = counts[labels[selected]] / budgets[labels[selected]]
    random_sample = np.random.default_rng(cfg.seed).choice(len(negative), len(selected), replace=False)
    table = pd.DataFrame({'cluster': np.arange(k), 'population': counts, 'selected': budgets,
                          'inclusion_probability': np.divide(budgets, counts, out=np.zeros(k, float), where=counts > 0)})
    table.to_csv(root / 'cluster_counts.csv', index=False)
    audits = []
    original = day.target_x[negative]
    for column in range(original.shape[1]):
        cuts = np.unique(np.quantile(reference[:, column], np.linspace(0, 1, 21)[1:-1]))
        bins = np.r_[-np.inf, cuts, np.inf]
        p = np.histogram(original[:, column], bins)[0].astype(float)
        p /= p.sum()
        for method, ids, weights in [('cluster_raw', selected, None), ('cluster_ipw', selected, ipw),
                                      ('random_same_budget', random_sample, None)]:
            q = np.histogram(original[ids, column], bins, weights=weights)[0].astype(float)
            q /= q.sum()
            ps = (p + 1e-8) / (p + 1e-8).sum(); qs = (q + 1e-8) / (q + 1e-8).sum()
            audits.append({'feature': column, 'method': method, 'KL_P_to_Q': float(np.sum(ps * np.log(ps / qs))),
                           'total_variation': float(np.abs(p - q).sum() / 2)})
    audit = pd.DataFrame(audits)
    audit.to_csv(root / 'cluster_distribution_audit.csv', index=False)
    # Fit projection on training references only, never validation/test.
    pca = PCA(n_components=2).fit(scaler.transform(reference))
    visible = np.random.default_rng(42).choice(len(negative), min(2000, len(negative)), replace=False)
    projection = pca.transform(scaler.transform(original[visible]))
    chosen = pca.transform(scaler.transform(original[selected[:2000]]))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].scatter(projection[:, 0], projection[:, 1], c=labels[visible], s=12, cmap='tab20', alpha=.35)
    axes[0].scatter(chosen[:, 0], chosen[:, 1], s=40, facecolors='none', edgecolors='black', label='Selected')
    axes[0].set_title('Training negatives: clusters and selected rows'); axes[0].legend()
    table.set_index('cluster')[['population', 'selected']].plot.bar(ax=axes[1], logy=True)
    axes[1].set_title('Cluster population vs selected (log scale)')
    audit.groupby('method').KL_P_to_Q.mean().plot.bar(ax=axes[2])
    axes[2].set_title('Mean feature KL: original -> sample')
    fig.tight_layout();fig.savefig(root / 'cluster_audit.png', dpi=160);plt.show()
    # Visualize actual transaction edges; keep parallel transfers as separate curves.
    ids = negative[selected[:30]]
    nodes = np.unique(np.r_[day.target_src[ids], day.target_dst[ids]])
    angles = np.linspace(0, 2 * np.pi, len(nodes), endpoint=False)
    positions = {int(node): (np.cos(t), np.sin(t)) for node, t in zip(nodes, angles)}
    fig, ax = plt.subplots(figsize=(7, 6))
    for order, position in enumerate(ids):
        src, dst = int(day.target_src[position]), int(day.target_dst[position])
        ax.annotate('', xy=positions[dst], xytext=positions[src],
                    arrowprops={'arrowstyle': '->', 'alpha': .45, 'connectionstyle': f'arc3,rad={.12 + .06 * (order % 4)}'})
    for node, (x, y) in positions.items():
        ax.scatter(x, y, s=180, color='lightblue', edgecolor='black');ax.text(x, y, str(node), ha='center', va='center', fontsize=8)
    ax.set_title('Up to 30 selected real transactions; circular layout');ax.axis('off')
    fig.tight_layout();fig.savefig(root / 'cluster_graph_sample.png', dpi=160);plt.show()
    checks = {'all_nonempty_clusters_preserved': bool((budgets[counts > 0] > 0).all()),
              'budget_within_population': bool((budgets <= counts).all()),
              'negative_ipw_total_matches_population': bool(np.isclose(ipw.sum(), len(negative))),
              'projection_train_only': True, 'test_used': False}
    if not all(checks[key] for key in checks if key != 'test_used'):
        raise AssertionError(checks)
    data.json_write(root / 'cluster_checks.json', checks)
    return table, audit, checks

## `extract_latent`

모델 가중치를 불러와 고정한 뒤 날짜/클래스별 상한만큼 표현을 추출합니다. 정답·거래 ID·날짜·추출 보정 가중치를 별도 표로 반환합니다.

**입력:** `name`, `seed`, `cfg`, `split`, `cap_per_class`

**반환:** `(np.concatenate(vectors).astype('float32'), pd.concat(records, ignore_index=True))`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def extract_latent(name, seed, cfg, split, cap_per_class=2048):
    """Freeze one GNN/head projection; extract stratified bounded latent banks."""
    root = Path(cfg.output).parent
    model = TransactionGNN(name, hidden=cfg.hidden, heads=cfg.heads).cpu()
    model.load_state_dict(torch.load(root / 'models' / f'{name}_seed{seed}' / 'model.pt', map_location='cpu', weights_only=True))
    model.eval()
    vectors, records = [], []
    for index, date in enumerate(graph_dates(cfg, split)):
        day = read_graph(cfg, date)
        if split == 'train':
            table = pd.read_parquet(root / 'samples' / f'{date}.parquet').copy()
        else:
            table = pd.DataFrame({'position': np.arange(len(day.labels)), 'edge_id': day.target_ids,
                                  'y': day.labels, 'sample_weight': np.ones(len(day.labels))})
        rng = np.random.default_rng(seed + index)
        parts = []
        for label in (0, 1):
            group = table[table.y == label]
            if len(group):
                count = min(cap_per_class, len(group))
                chosen = group.iloc[rng.choice(len(group), count, replace=False)].copy()
                chosen['sample_weight'] *= len(group) / count
                parts.append(chosen)
        table = pd.concat(parts).sort_values('position').reset_index(drop=True)
        table['day'] = date
        sampler = data.FixedNeighborSampler(day)
        with torch.no_grad():
            for start in range(0, len(table), cfg.target_batch_size):
                ix = table.position.to_numpy()[start:start + cfg.target_batch_size]
                x, ei, ea, targets, ta = data.tensor_batch(sampler.batch(ix), 'cpu')
                hidden = torch.relu(model.conv1(x, ei, ea));hidden = torch.relu(model.conv2(hidden, ei, ea))
                joined = torch.cat([hidden[targets[0]], hidden[targets[1]], ta], dim=1)
                vectors.append(model.head[:2](joined).numpy())
        records.append(table)
    return np.concatenate(vectors).astype('float32'), pd.concat(records, ignore_index=True)

## `latent_smote`

학습 이상 표현을 보간해 추가 표현을 만듭니다. 생성값과 함께 왼쪽/오른쪽 이웃 번호·보간 비율을 반환하므로 수식을 직접 검증할 수 있습니다.

**입력:** `bank`, `count`, `seed`, `k`

**반환:** `(synthetic, {'left': left, 'right': right, 'alpha': alpha})`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def latent_smote(bank, count, seed=42, k=5):
    """Interpolate positive training vectors and return exact construction evidence."""
    if len(bank) < 2:
        raise ValueError('SMOTE needs at least two training positives')
    k = min(k, len(bank) - 1)
    neighbors = NearestNeighbors(n_neighbors=k + 1).fit(bank).kneighbors(bank, return_distance=False)
    neighbors = np.stack([row[row != index][:k] for index, row in enumerate(neighbors)])
    rng = np.random.default_rng(seed)
    left = rng.integers(len(bank), size=count)
    right = neighbors[left, rng.integers(k, size=count)]
    alpha = rng.random((count, 1)).astype('float32')
    synthetic = bank[left] + alpha * (bank[right] - bank[left])
    assert (left != right).all()
    return synthetic, {'left': left, 'right': right, 'alpha': alpha}

## `gradient_penalty`

진짜와 생성 표현 사이의 보간점에서 critic의 입력 기울기를 구하고 그 크기와 1의 차이를 벌점으로 계산합니다.

**입력:** `critic`, `real`, `fake`

**반환:** `((gradient.norm(2, dim=1) - 1) ** 2).mean()`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def gradient_penalty(critic, real, fake):
    """Penalize deviation of interpolated-input gradient norm from one."""
    alpha = torch.rand(len(real), 1)
    mixed = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    score = critic(mixed)
    gradient = torch.autograd.grad(score, mixed, torch.ones_like(score), create_graph=True)[0]
    return ((gradient.norm(2, dim=1) - 1) ** 2).mean()

## `latent_wgan`

학습 이상 표현만 사용해 WGAN-GP를 학습합니다. 생성한 표현과 스텝별 손실·기울기 벌점 표를 반환합니다.

**입력:** `bank`, `count`, `seed`, `steps`

**반환:** `(synthetic, pd.DataFrame(history))`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def latent_wgan(bank, count, seed=42, steps=100):
    """Train a small WGAN-GP only on standardized positive training vectors."""
    torch.manual_seed(seed)
    scaler = StandardScaler().fit(bank)
    real_bank = torch.tensor(scaler.transform(bank), dtype=torch.float32)
    dim = bank.shape[1]
    generator = torch.nn.Sequential(torch.nn.Linear(32, 128), torch.nn.LeakyReLU(.2),
        torch.nn.Linear(128, 128), torch.nn.LeakyReLU(.2), torch.nn.Linear(128, dim))
    critic = torch.nn.Sequential(torch.nn.Linear(dim, 128), torch.nn.LeakyReLU(.2),
        torch.nn.Linear(128, 128), torch.nn.LeakyReLU(.2), torch.nn.Linear(128, 1))
    go = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0., .9))
    co = torch.optim.Adam(critic.parameters(), lr=1e-4, betas=(0., .9))
    history = []
    for step in range(steps):
        for _ in range(3):
            real = real_bank[torch.randint(len(bank), (min(64, len(bank)),))]
            fake = generator(torch.randn(len(real), 32)).detach()
            co.zero_grad(set_to_none=True)
            penalty = gradient_penalty(critic, real, fake)
            critic_loss = critic(fake).mean() - critic(real).mean() + 10 * penalty
            critic_loss.backward();co.step()
        go.zero_grad(set_to_none=True)
        generator_loss = -critic(generator(torch.randn(len(real), 32))).mean()
        generator_loss.backward();go.step()
        if not torch.isfinite(critic_loss + generator_loss):
            raise ValueError('Nonfinite GAN loss')
        history.append({'step': step + 1, 'critic_loss': float(critic_loss.detach()),
                        'generator_loss': float(generator_loss.detach()), 'gradient_penalty': float(penalty.detach())})
    with torch.no_grad():
        synthetic = scaler.inverse_transform(generator(torch.randn(count, 32)).numpy()).astype('float32')
    return synthetic, pd.DataFrame(history)

## `synthetic_diagnostics`

생성 표현의 유한성·분산·최근접 거리·복제율을 계산합니다. 붕괴 의심 여부는 참고 진단이며 통과/실패 성능 판정이 아닙니다.

**입력:** `real`, `synthetic`

**반환:** `{'finite': bool(np.isfinite(synthetic).all()), 'median_std_ratio': ratio, 'median_nearest_real_distance': float(np.median(distances)), 'exac`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def synthetic_diagnostics(real, synthetic):
    """Report variance coverage and nearest-real distances, including collapse warning."""
    active = real.std(0) > 1e-6
    ratio = float(np.median(synthetic.std(0)[active] / real.std(0)[active])) if active.any() else 0.
    distances = NearestNeighbors(n_neighbors=1).fit(real).kneighbors(synthetic, return_distance=True)[0][:, 0]
    return {'finite': bool(np.isfinite(synthetic).all()), 'median_std_ratio': ratio,
            'median_nearest_real_distance': float(np.median(distances)),
            'exact_duplicate_fraction': float(np.mean(distances < 1e-6)),
            'possible_collapse': bool(ratio < .05 or np.mean(distances < 1e-6) > .95)}

## `balanced_loss`

반복/합성 표본 수와 무관하게 클래스별 총 손실 비중을 고정합니다.

**입력:** `negative_logits`, `positive_logits`, `alpha`

**반환:** `(torch.nn.functional.softplus(negative_logits).mean() + alpha * torch.nn.functional.softplus(-positive_logits).mean()) / (1 + alpha)`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def balanced_loss(negative_logits, positive_logits, alpha):
    """Keep the total positive loss mass equal across repeat/SMOTE/GAN controls."""
    return (torch.nn.functional.softplus(negative_logits).mean() +
            alpha * torch.nn.functional.softplus(-positive_logits).mean()) / (1 + alpha)

## `fit_head`

고정 표현으로 새 분류기 하나를 학습합니다. 검증 F1로 임계값을 정하고 추출 보정된 검증 AP·F1 등을 반환합니다.

**입력:** `train_x`, `train_table`, `val_x`, `val_table`, `synthetic`, `seed`, `steps`, `pos_weight`

**반환:** `{'AP': float(average_precision_score(y, scores, sample_weight=w)), 'precision': float(precision_score(y, pred, sample_weight=w, zero_divisio`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def fit_head(train_x, train_table, val_x, val_table, synthetic, seed, steps, pos_weight):
    """Fit an identically initialized fresh classifier with equal update budget."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    negative = train_x[train_table.y.to_numpy() == 0]
    positive = train_x[train_table.y.to_numpy() == 1]
    negative_w = train_table.loc[train_table.y == 0, 'sample_weight'].to_numpy()
    positive_w = train_table.loc[train_table.y == 1, 'sample_weight'].to_numpy()
    alpha = pos_weight * positive_w.sum() / negative_w.sum()
    if synthetic is None:
        pool, probabilities = positive, positive_w / positive_w.sum()
    else:
        pool = np.concatenate([positive, synthetic])
        probabilities = np.r_[.5 * positive_w / positive_w.sum(), np.full(len(synthetic), .5 / len(synthetic))]
    dim = train_x.shape[1]
    head = torch.nn.Sequential(torch.nn.Linear(dim, 128), torch.nn.ReLU(), torch.nn.Linear(128, 1))
    optimizer = torch.optim.Adam(head.parameters(), lr=.001)
    for _ in range(steps):
        nx = torch.tensor(negative[rng.choice(len(negative), 64, p=negative_w / negative_w.sum())])
        px = torch.tensor(pool[rng.choice(len(pool), 64, p=probabilities)])
        optimizer.zero_grad(set_to_none=True)
        loss = balanced_loss(head(nx), head(px), alpha)
        if not torch.isfinite(loss):
            raise ValueError('Nonfinite classifier loss')
        loss.backward();optimizer.step()
    with torch.no_grad():
        scores = torch.sigmoid(head(torch.tensor(val_x))).numpy().ravel()
    y, w = val_table.y.to_numpy(), val_table.sample_weight.to_numpy()
    precision, recall, thresholds = precision_recall_curve(y, scores, sample_weight=w)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    threshold = float(thresholds[np.argmax(f1)])
    pred = scores >= threshold
    return {'AP': float(average_precision_score(y, scores, sample_weight=w)),
            'precision': float(precision_score(y, pred, sample_weight=w, zero_division=0)),
            'recall': float(recall_score(y, pred, sample_weight=w, zero_division=0)),
            'F1': float(f1_score(y, pred, sample_weight=w, zero_division=0)),
            'threshold': threshold, 'positive_loss_mass_alpha': float(alpha)}

## `visualize_synthesis`

같은 학습 양성 PCA 좌표계에서 실제/추가 표현을 비교하고 GAN 손실을 그립니다. 그림을 PNG로 저장합니다.

**입력:** `real`, `pools`, `history`, `folder`, `name`

**반환:** 그림 표시와 파일 저장

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def visualize_synthesis(real, pools, history, folder, name):
    """Use one train-fitted PCA for all methods and display WGAN diagnostics."""
    folder.mkdir(parents=True, exist_ok=True)
    pca = PCA(n_components=2).fit(real)
    original = pca.transform(real)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, method in zip(axes, ('repeat', 'smote', 'wgan_gp')):
        generated = pca.transform(pools[method])
        ax.scatter(original[:2000, 0], original[:2000, 1], s=15, alpha=.6, label='Real training positives')
        ax.scatter(generated[:2000, 0], generated[:2000, 1], s=12, alpha=.4, label='Added latent vectors')
        ax.set_title(name + ': ' + method);ax.legend(fontsize=7)
    fig.tight_layout();fig.savefig(folder / f'{name}_latent_pca.png', dpi=160);plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    history.plot(x='step', y=['critic_loss', 'generator_loss'], ax=axes[0]);axes[0].set_title('WGAN training losses')
    history.plot(x='step', y='gradient_penalty', ax=axes[1]);axes[1].set_title('Gradient penalty')
    fig.tight_layout();fig.savefig(folder / f'{name}_gan_losses.png', dpi=160);plt.show()

## `augmentation_comparison`

모델·시드마다 none/repeat/SMOTE/WGAN-GP를 같은 조건으로 비교합니다. 시각화, 진단표, 데이터 분리 검사와 성능 표를 저장합니다.

**입력:** `settings`, `cfg`, `gan_steps`, `head_steps`, `cap_per_class`

**반환:** `(frame, diagnostic_frame, pd.DataFrame(checks))`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def augmentation_comparison(settings, cfg, gan_steps=100, head_steps=100, cap_per_class=2048):
    """Compare none/repeat/SMOTE/WGAN for every requested model and seed."""
    torch.set_num_threads(cfg.threads)
    folder = Path(cfg.output).parent / 'analysis' / 'augmentation'
    folder.mkdir(parents=True, exist_ok=True)
    rows, diagnostics, checks = [], [], []
    for name in settings['models']:
        for seed in settings['seeds']:
            train_x, train_table = extract_latent(name, seed, cfg, 'train', cap_per_class)
            val_x, val_table = extract_latent(name, seed, cfg, 'val', cap_per_class)
            assert set(train_table.edge_id).isdisjoint(val_table.edge_id)
            assert ((train_table.day >= cfg.train_start) & (train_table.day < cfg.val_start)).all()
            assert ((val_table.day >= cfg.val_start) & (val_table.day < cfg.test_start)).all()
            if train_table.y.nunique() != 2 or val_table.y.nunique() != 2:
                raise ValueError('Train/validation latent banks need both classes')
            positive = train_x[train_table.y.to_numpy() == 1]
            rng = np.random.default_rng(seed)
            positive_w = train_table.loc[train_table.y == 1, 'sample_weight'].to_numpy()
            count = len(positive)
            repeated = positive[rng.choice(len(positive), count, p=positive_w / positive_w.sum())]
            smote, evidence = latent_smote(positive, count, seed)
            gan, history = latent_wgan(positive, count, seed, gan_steps)
            pools = {'repeat': repeated, 'smote': smote, 'wgan_gp': gan}
            expected = positive[evidence['left']] + evidence['alpha'] * (positive[evidence['right']] - positive[evidence['left']])
            np.testing.assert_allclose(smote, expected, atol=1e-6)
            for method, values in pools.items():
                assert values.shape == positive.shape and np.isfinite(values).all()
                diagnostics.append({'model': name, 'seed': seed, 'method': method, **synthetic_diagnostics(positive, values)})
            for method in ('none', 'repeat', 'smote', 'wgan_gp'):
                result = fit_head(train_x, train_table, val_x, val_table, pools.get(method), seed, head_steps, cfg.pos_weight)
                rows.append({'model': name, 'seed': seed, 'method': method, 'validation_rows': len(val_table),
                             'cap_per_class_per_day': cap_per_class, **result})
            checks.append({'model': name, 'seed': seed, 'split_disjoint': True,
                           'smote_segment_verified': True, 'same_synthetic_budget': True, 'test_used': False})
            if seed == settings['seeds'][0]:
                visualize_synthesis(positive, pools, history, folder, name)
            history.to_csv(folder / f'{name}_seed{seed}_gan_history.csv', index=False)
            print('Latent comparison completed:', name, seed, flush=True)
    frame = pd.DataFrame(rows)
    control = frame[frame.method == 'none'].set_index(['model', 'seed']).AP
    frame['AP_gain_pp'] = [100 * (row.AP - control.loc[(row.model, row.seed)]) for row in frame.itertuples()]
    frame.to_csv(folder / 'comparison.csv', index=False)
    diagnostic_frame = pd.DataFrame(diagnostics);diagnostic_frame.to_csv(folder / 'diagnostics.csv', index=False)
    data.json_write(folder / 'checks.json', checks)
    summary = frame.groupby(['model', 'method'])[['AP', 'F1', 'AP_gain_pp']].mean()
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    summary.AP.unstack().plot.bar(ax=axes[0]);axes[0].set_title('Matched latent-head validation AP')
    summary.AP_gain_pp.unstack().plot.bar(ax=axes[1]);axes[1].axhline(0, color='black', linewidth=.5);axes[1].set_title('AP change vs none (percentage points)')
    fig.tight_layout();fig.savefig(folder / 'method_comparison.png', dpi=160);plt.show()
    return frame, diagnostic_frame, pd.DataFrame(checks)

## `run_method_tests`

군집 예산·SMOTE 선분/재현성·반복 손실 불변성·WGAN-GP 역전파·붕괴 탐지 단위 검사를 수행합니다.

**입력:** 

**반환:** `pd.DataFrame(tests)`

아래 셀은 정의만 수행합니다. 실제 호출은 통합 실행의 학습 다음 단계에 있습니다.

In [ ]:
def run_method_tests():
    """Check allocation, SMOTE geometry, constant class mass and GP backward."""
    tests = []
    counts = np.array([0, 1, 10, 100, 1000])
    budget = allocate(counts, .1)
    assert budget.sum() == max(np.count_nonzero(counts), int(np.ceil(counts.sum() * .1)))
    assert (budget <= counts).all() and (budget[counts > 0] > 0).all()
    tests.append({'test': 'cluster budget, minimum one, capacity', 'passed': True})
    bank = np.arange(20, dtype='float32')[:, None] * np.array([[1., 2., 3.]], dtype='float32')
    synthetic, evidence = latent_smote(bank, 100)
    np.testing.assert_allclose(synthetic[:, 1], synthetic[:, 0] * 2, atol=1e-5)
    assert (evidence['left'] != evidence['right']).all()
    np.testing.assert_array_equal(synthetic, latent_smote(bank, 100)[0])
    tests.append({'test': 'SMOTE segment, no self-neighbor, seed repeatability', 'passed': True})
    negative, positive = torch.tensor([-1., .5]), torch.tensor([.1, 2.])
    torch.testing.assert_close(balanced_loss(negative, positive, .7), balanced_loss(negative, positive.repeat(2), .7))
    tests.append({'test': 'repetition does not increase positive loss mass', 'passed': True})
    critic = torch.nn.Sequential(torch.nn.Linear(3, 8), torch.nn.LeakyReLU(.2), torch.nn.Linear(8, 1))
    penalty = gradient_penalty(critic, torch.randn(8, 3), torch.randn(8, 3));penalty.backward()
    assert torch.isfinite(penalty) and all(torch.isfinite(p.grad).all() for p in critic.parameters() if p.grad is not None)
    tests.append({'test': 'WGAN-GP finite gradient and backward', 'passed': True})
    collapsed = np.tile(bank.mean(0), (100, 1))
    assert synthetic_diagnostics(bank, collapsed)['possible_collapse']
    tests.append({'test': 'constant-output collapse detected', 'passed': True})
    return pd.DataFrame(tests)

## 통합 실행에 연결

In [ ]:
export_cells('50_클러스터링과증강_시각화.ipynb', 'analysis')